### Raw data Loading

In [0]:
tags_path = "/Volumes/workspace/default/moviedata/tags.csv"
ratings_path = "/Volumes/workspace/default/moviedata/ratings.csv"
movies_path = "/Volumes/workspace/default/moviedata/movies.csv"
links_path = "/Volumes/workspace/default/moviedata/links.csv"

In [0]:
df_tags = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(tags_path)

df_ratings = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(ratings_path)

df_movies = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(movies_path)

df_links = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(links_path)

In [0]:
df_tags.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/moviedata_delta/tags")

df_ratings.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/moviedata_delta/ratings")

df_movies.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/moviedata_delta/movies")

df_links.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/moviedata_delta/links")

In [0]:
df_tags_delta = spark.read.format("delta").load("/Volumes/workspace/default/moviedata_delta/tags")

df_ratings_delta = spark.read.format("delta").load("/Volumes/workspace/default/moviedata_delta/ratings")

df_movies_delta = spark.read.format("delta").load("/Volumes/workspace/default/moviedata_delta/movies")

df_links_delta = spark.read.format("delta").load("/Volumes/workspace/default/moviedata_delta/links")

In [0]:
df_tags_delta.display()

In [0]:
from pyspark.sql.functions import col, from_unixtime

df_tags_clean = df_tags_delta

# Step 1 — datatype conversion
df_tags_clean = df_tags_clean.selectExpr(
    "cast(userId as int) as userId",
    "cast(movieId as int) as movieId",
    "tag",
    "cast(timestamp as long) as timestamp"
)

# Step 2 — remove null values
df_tags_clean = df_tags_clean.filter(
    col("tag").isNotNull() &
    col("timestamp").isNotNull()
)

# Step 3 — remove duplicates
df_tags_clean = df_tags_clean.dropDuplicates(
    ["userId","movieId","tag","timestamp"]
)

# Step 4 — convert unix timestamp → readable timestamp
df_tags_clean = df_tags_clean.withColumn(
    "tag_timestamp",
    from_unixtime(col("timestamp"))
)

# optional drop old column
df_tags_clean = df_tags_clean.drop("timestamp")

# Step 5 — save Silver table
df_tags_clean.write.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.save("/Volumes/workspace/default/silver/tags")

# preview result
df_tags_clean.display()

In [0]:
df_ratings_delta.display()

In [0]:
from pyspark.sql.functions import expr, col, from_unixtime

# Step 1 — safe datatype conversion using try_cast
df_ratings_clean = df_ratings_delta.selectExpr(
    "cast(userId as int) as userId",
    "cast(movieId as int) as movieId",
    "try_cast(rating as double) as rating",
    "cast(timestamp as long) as timestamp"
)

# Step 2 — remove duplicate rows
df_ratings_clean = df_ratings_clean.dropDuplicates(
    ["userId","movieId","timestamp"]
)

# Step 3 — remove null ratings
df_ratings_clean = df_ratings_clean.dropna(
    subset=["rating"]
)

# Step 4 — ensure rating between 0 and 5
df_ratings_clean = df_ratings_clean.filter(
    col("rating").between(0,5)
)

# Step 5 — convert unix timestamp → readable timestamp
df_ratings_clean = df_ratings_clean.withColumn(
    "rating_timestamp",
    from_unixtime(col("timestamp"))
)

# Step 6 — drop old timestamp column
df_ratings_clean = df_ratings_clean.drop("timestamp")

# Step 7 — save as Silver Delta table
df_ratings_clean.write.format("delta") \
.mode("overwrite") \
.save("/Volumes/workspace/default/silver/ratings")

# Step 8 — display result
df_ratings_clean.display()

In [0]:
(df_ratings_delta.count(), len(df_ratings_delta.columns))

In [0]:
(df_ratings_clean.count(), len(df_ratings_clean.columns))

In [0]:
df_movies_delta.display()

In [0]:
df_movies_clean = df_movies_delta.dropDuplicates()

In [0]:
df_movies_clean.display()

In [0]:
from pyspark.sql.functions import col, trim

# Step 1 — clean datatype using try_cast
df_movies_clean = df_movies_delta.selectExpr(
    "cast(movieId as int) as movieId",
    "title",
    "genres"
)

# Step 2 — convert string 'null' → actual null
df_movies_clean = df_movies_clean \
.withColumn("title",
    trim(col("title"))
) \
.withColumn("genres",
    trim(col("genres"))
)

# Step 3 — remove rows where title OR genres is null
df_movies_clean = df_movies_clean.filter(
    col("title").isNotNull() &
    col("genres").isNotNull()
)

# Step 4 — remove rows where title OR genres contains string 'null'
df_movies_clean = df_movies_clean.filter(
    (col("title") != "null") &
    (col("genres") != "null")
)

# Step 5 — save to Silver layer
df_movies_clean.write.format("delta") \
.mode("overwrite") \
.save("/Volumes/workspace/default/silver/movies")

# Step 6 — preview cleaned data
df_movies_clean.display()

In [0]:
df_links_delta.display()

In [0]:
from pyspark.sql.functions import *

df_links_clean = df_links_delta

# Step 1 — convert 'unknown' → null
df_links_clean = df_links_clean.withColumn(
    "imdbId",
    when(trim(col("imdbId")) == "unknown", None)
    .otherwise(col("imdbId"))
)

# Step 2 — datatype conversion
df_links_clean = df_links_clean.selectExpr(
    "cast(movieId as int) as movieId",
    "try_cast(imdbId as int) as imdbId",
    "try_cast(tmdbId as int) as tmdbId"
)

# Step 3 — remove null values (important for joins)
df_links_clean = df_links_clean.filter(
    col("movieId").isNotNull() &
    col("imdbId").isNotNull() &
    col("tmdbId").isNotNull()
)

# Step 4 — remove duplicates
df_links_clean = df_links_clean.dropDuplicates(
    ["movieId"]
)

# Step 5 — save Silver layer
df_links_clean.write.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.save("/Volumes/workspace/default/silver/links")

# preview
df_links_clean.display()